# 03 — A címkék ellenőrzése / Validating the labels

HU: Valódi címke hiányában öt bizonyítékot gyűjtünk arról, hogy a BERT-címkék megbízhatóak:

| # | Bizonyíték / Evidence | Mit mutat? / What does it show? |
|---|---|---|
| 1 | Eloszlások / Distributions | mennyire térnek el a források / how the sources differ |
| 2 | Egyezés (kappa) / Agreement | a három megközelítés konzisztenciája / consistency of the three |
| 3 | Vendégpontszám / Guest score | konvergens validitás / convergent validity |
| 4 | Mezőpróba, HRAST-teszt / Field test, HRAST test | független ellenőrzés / independent checks |
| 5 | **Kézi gold minta / Manual gold sample** | **közvetlen pontosság / direct accuracy** |

Ez a notebook CPU-n, helyben fut (a 02 mentett eredményeiből). / Runs locally on a CPU (from the saved outputs of 02).

In [ ]:
# HU: A notebooks/ mappából futunk, ezért a szülőmappát a path-hoz adjuk (src csomag).
# EN: We run from the notebooks/ folder, so the parent folder is added to the path (src package).
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import set_seed
from src.config import CFG, PATHS, SENTIMENT_COLORS, INK, APPROACHES, apply_plot_style
from src import data_loading as dl
from src import preprocessing as pp
from src import evaluation as ev
from src import label_validation as lv
from src import pipeline

# HU: Reprodukálhatóság + egységes ábrastílus. / EN: Reproducibility + uniform figure style.
set_seed()
PATHS.ensure()
apply_plot_style()
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 170)

data = dl.read_frame(pipeline.LABELLED).set_index("review_id")
data["score_cut_label"] = lv.score_cut_labels(data["score"])     # HU/EN: REFERENCE ONLY
approaches = lv.available_approaches(data)
SOURCES = {**{f"{a}": f"{a}_label" for a in approaches}, "final": "label_final",
           "score cut (reference)": "score_cut_label"}
print(f"{len(data):,} reviews | approaches: {approaches}")

## 1. Eloszlások / Distributions

In [ ]:
dist = lv.label_distribution(data, SOURCES)
display(dist)
ev.save_table(dist, "03_label_distribution")
lv.plot_label_distribution(dist, title="Class distribution by label source", save_as="03_label_distribution")
plt.show()

## 2. Egyezés a megközelítések között / Agreement between the approaches

HU: A (C) a saját tanítóvéleményein a tanárokat "visszamondja", ezért azokat kizárjuk (`comparison_mask`).
EN: On its own training reviews (C) "repeats" the teachers, so those are excluded (`comparison_mask`).

In [ ]:
mask = lv.comparison_mask(data)
print(f"Rows used for agreement: {mask.sum():,} of {len(data):,}")
agreement = lv.pairwise_agreement(data[mask], SOURCES)
display(agreement)
ev.save_table(agreement, "03_pairwise_agreement")
fk = lv.fleiss_kappa(data[mask], [f"{a}_label" for a in approaches])
print(f"Fleiss' kappa ({', '.join(approaches)}): {fk:.3f}")

In [ ]:
pairs = [("A_label", "B_label"), ("A_label", "C_label"), ("label_final", "score_cut_label")]
for col_a, col_b in pairs:
    if {col_a, col_b} <= set(data.columns):
        tab = lv.approach_crosstab(data[mask], col_a, col_b)
        name = f"03_crosstab_{col_a}_vs_{col_b}"
        ev.save_table(tab, name)
        lv.plot_agreement_matrix(tab, f"{col_a} vs {col_b}", xlabel=col_b, ylabel=col_a, save_as=name)
        plt.show()

## 3. Kapcsolat a vendégpontszámmal / Relation to the guest score

HU: Erős, de **nem tökéletes** kapcsolat a jó eredmény. A `neg_but_score_ge9` oszlop mutatja, hányszor panaszkodik a
szöveg magas pontszám mellett — ezt a pontszám-vágás nem látná.
EN: A strong but **imperfect** relation is the good outcome. `neg_but_score_ge9` shows how often the text complains
despite a high score — a score cut would miss this.

In [ ]:
# HU: A végső címkét is "megközelítésként" kezeljük. / EN: The final label is treated as an "approach" too.
data["final_label"], data["final_score_cont"] = data["label_final"], data["score_cont_final"]
validity = lv.guest_score_validity(data, approaches + ["final"])
display(validity)
ev.save_table(validity, "03_guest_score_validity")
lv.plot_score_vs_guest(data, approaches + ["final"], save_as="03_text_score_vs_guest_score")
plt.show()

In [ ]:
# HU: Ahol a szöveg és a pontszám nagyon eltér — olvasd el, és válassz idézeteket a dolgozatba.
# EN: Where text and score differ strongly — read them and pick quotes for the thesis.
disc = lv.text_score_discrepancies(data.reset_index(), min_gap=4, n_examples=12)
ev.save_table(disc, "03_text_score_discrepancies")
for _, r in disc.iterrows():
    print(f"[guest {r['score']:.0f} | text {r['score10_final']} | {r['label_final']}]  {r['text'][:230]}\n")

## 4. Független próbák / Independent checks

In [ ]:
for name in ["02_field_test", "02_approach_b_hrast_test"]:
    path = PATHS.tables / f"{name}.csv"
    if path.exists():
        print(f"=== {name} ===")
        display(pd.read_csv(path))
    else:
        print(f"{name}: not available (run notebook 02).")

## 5. Kézi gold minta / Manual gold sample

HU: Az **egyetlen közvetlen pontosságmérés**. Lépések:
1. az alábbi cella exportálja a `data/annotation/gold_sample_to_annotate.csv` fájlt (csak szöveg — **vak** annotálás);
2. töltsd ki a `gold_label` (negative / neutral / positive) és a `gold_score` (1–10) oszlopot a `docs/ANNOTATION_GUIDE.md` szerint;
3. futtasd újra a cellát → összevetés minden forrással, a **pontszám-vágással együtt**.

EN: The **only direct accuracy measurement**. Export → annotate blind (`docs/ANNOTATION_GUIDE.md`) → re-run → comparison
with every source, **including the score cut**.

In [ ]:
gold_file = PATHS.annotation / "gold_sample_to_annotate.csv"
if not gold_file.exists():
    lv.export_gold_sample(data.reset_index())
    print("Exported. Annotate it, then re-run this cell.")
else:
    gold = lv.load_gold()
    if len(gold) == 0:
        print("The gold file exists but has no valid gold_label yet.")
    else:
        score_cols = {**{a: f"{a}_score_cont" for a in approaches}, "final": "score_cont_final",
                      "score cut (reference)": "score"}
        gold_eval = lv.evaluate_on_gold(gold, data, SOURCES, score_cols)
        display(gold_eval)
        ev.save_table(gold_eval, "03_gold_evaluation")
        print("=> Set CFG.tie_breaker_approach in config.py to the best approach.")